In [ ]:
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot
from cellrank import pl as cr_pl
from cellrank.estimators import GPCCA
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel
from cellrank.models import GAM

series_name = "macnair_validation_oligodendrocyte"
clustered_data_location = find_env_dir("CLUSTERED_DATA")

filtered_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
cellrank_analysis_location = find_env_dir("CELLRANK")
cellrank_analysis_location = os.path.join(
    cellrank_analysis_location, series_name
)
os.makedirs(cellrank_analysis_location, exist_ok=True)

In [3]:
# Calculating connectivity graph between leiden clusters using PAGA
sc.tl.paga(filtered_adata, groups="leiden")
# Calculating Diffusion Pseudotime (DPT) for ordering cells along developmental trajectories
root_cell_index = np.where(filtered_adata.obs["leiden"] == "1")[0][0]
filtered_adata.uns["iroot"] = root_cell_index
sc.tl.diffmap(filtered_adata)
sc.tl.dpt(filtered_adata)

In [4]:
# CellRank-based Trajectory Inference and Gene Trend Analysis
pseudotime_kernel = PseudotimeKernel(filtered_adata, time_key='dpt_pseudotime')
pseudotime_kernel.compute_transition_matrix()


  0%|          | 0/151592 [00:00<?, ?cell/s]

PseudotimeKernel[n=151592, dnorm=False, scheme='hard', frac_to_keep=0.3]

In [5]:
connectivity_kernel = ConnectivityKernel(filtered_adata)
connectivity_kernel.compute_transition_matrix()

ConnectivityKernel[n=151592, dnorm=True, key='connectivities']

In [6]:
kernel = 0.8 * pseudotime_kernel + 0.2 * connectivity_kernel
kernel.compute_transition_matrix()
kernel.plot_projection(basis='umap', color='leiden', title='Model-based Differentiation Flow', 
                       save = os.path.join(cellrank_analysis_location, "differentiation_flow_umap.svg")) 

saving figure to file /home/sjkim/protocols/results/cellrank/macnair_validation_oligodendrocyte/differentiation_flow_umap.svg


In [7]:
#  Generalized Perron Cluster Cluster Analysis (GPCCA)
gpcca = GPCCA(kernel)
gpcca.fit(cluster_key="leiden", n_states=None)
gpcca.predict_terminal_states()
gpcca.predict_initial_states(allow_overlap=True)

gpcca.compute_fate_probabilities() #type: ignore
gpcca.plot_fate_probabilities( #type: ignore
        same_plot=True,
        basis="umap",
        save=os.path.join(cellrank_analysis_location, "fate_probabilities_umap.svg"),
    )

  0%|          | 0/1 [00:00<?, ?/s]

saving figure to file /home/sjkim/protocols/results/cellrank/macnair_validation_oligodendrocyte/fate_probabilities_umap.svg


In [ ]:
target_gene = "TCP11L2"
trend_model = GAM(filtered_adata)
lineages = gpcca.terminal_states.cat.categories #type: ignore
cr_pl.gene_trends(
    adata,
    model=trend_model,
    genes=[target_gene],
    time_key="dpt_pseudotime",
    lineages=lineages,  # type: ignore
    data_key="X",
    same_plot=True,
    hide_cells=True,
    figsize=(10, 6),
    save=os.path.join(cellrank_analysis_location, f"gene_trends_{target_gene}.svg"),
)



In [10]:
from  pipeline.utils.cellrank import cellrank_analysis

cellrank_analysis(filtered_adata, series_name, start_cluster="1", target_gene="ANXA2", group_key="leiden")

  0%|          | 0/151592 [00:00<?, ?cell/s]

saving figure to file /home/sjkim/protocols/results/cellrank/macnair_validation_oligodendrocyte/differentiation_flow_umap.svg


  0%|          | 0/1 [00:00<?, ?/s]

saving figure to file /home/sjkim/protocols/results/cellrank/macnair_validation_oligodendrocyte/fate_probabilities_umap.svg


  0%|          | 0/1 [00:00<?, ?gene/s]

did not converge
             3_corr  3_pval  3_qval  3_ci_low  3_ci_high
C11orf96   0.225883     0.0     0.0  0.221101   0.230655
ANXA2      0.223041     0.0     0.0  0.218252   0.227819
FGFR1      0.218513     0.0     0.0  0.213714   0.223301
UHRF2      0.213606     0.0     0.0  0.208796   0.218405
CAMK2D     0.209162     0.0     0.0  0.204343   0.213970
TNFRSF12A  0.199301     0.0     0.0  0.194462   0.204130
NAV2       0.192957     0.0     0.0  0.188106   0.197799
IL6R       0.191910     0.0     0.0  0.187057   0.196754
MAP2K3     0.189895     0.0     0.0  0.185038   0.194743
PLA2G4C    0.186860     0.0     0.0  0.181997   0.191713


In [12]:
sc.pl.violin(
    filtered_adata,
    keys="dpt_pseudotime",
    groupby="leiden",
    rotation=0,
    stripplot=False
)